# Make Training Fit: Memory, Speed and Two GPUs

LLM Training Lab, module 8. Run this on Kaggle with **Settings > Accelerator > GPU T4 x2**. Parts 1 and 3 also run on one Colab T4; part 2 needs two GPUs.

Every cell matches a step on the lab page. Our recorded outputs are in `runs/lltMemoryThroughput/` of this repository.

In [ ]:
# Runtime check. Run this first. It prints the menu path that fixes each problem it finds.
import os, shutil, sys
import torch

problems = []
if sys.version_info < (3, 11):
    problems.append(f"Python {sys.version.split()[0]} is too old. Kaggle and Colab images ship 3.11 or later.")
if not torch.cuda.is_available():
    problems.append("No GPU. Kaggle: Settings > Accelerator > GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
else:
    names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    print("GPUs:", names, "| capability", torch.cuda.get_device_capability(0))
    if any("P100" in n for n in names):
        problems.append("P100 found. Current PyTorch wheels dropped it. Kaggle: Settings > Accelerator > GPU T4 x2.")
    if len(names) < 2:
        print("One GPU. Part 2 (two GPUs) needs Kaggle: Settings > Accelerator > GPU T4 x2. Parts 1 and 3 run here.")
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
disk = shutil.disk_usage("/tmp").free / 1e9
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | RAM {ram:.0f} GB | free disk in /tmp {disk:.0f} GB")
if disk < 20:
    problems.append("Under 20 GB free in /tmp. Step 10 writes a 16.2 GB checkpoint there.")
for p in problems or ["Runtime OK"]:
    print(p)


## Part 1. Memory and speed on one T4

In [ ]:
%%writefile gpt.py
# gpt.py - GPT-2 small (124M) in plain PyTorch, with the two memory switches this lab measures.
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class GPTConfig:
    vocab: int = 50257
    ctx: int = 1024
    layers: int = 12
    heads: int = 12
    d: int = 768


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.heads = c.heads
        self.ln1, self.ln2 = nn.LayerNorm(c.d), nn.LayerNorm(c.d)
        self.qkv, self.proj = nn.Linear(c.d, 3 * c.d), nn.Linear(c.d, c.d)
        self.fc, self.out = nn.Linear(c.d, 4 * c.d), nn.Linear(4 * c.d, c.d)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q, k, v = (t.view(B, T, self.heads, -1).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(y.transpose(1, 2).reshape(B, T, C))
        return x + self.out(F.gelu(self.fc(self.ln2(x)), approximate="tanh"))


def chunk_loss(h, w, y):
    return F.cross_entropy((h @ w.t()).float(), y, reduction="sum")


class GPT(nn.Module):
    def __init__(self, c, ckpt=False, chunks=0):
        super().__init__()
        self.ckpt, self.chunks = ckpt, chunks   # activation checkpointing, chunked cross-entropy
        self.wte, self.wpe = nn.Embedding(c.vocab, c.d), nn.Embedding(c.ctx, c.d)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.layers)])
        self.ln_f = nn.LayerNorm(c.d)
        self.head = nn.Linear(c.d, c.vocab, bias=False)
        self.head.weight = self.wte.weight          # tied, as in GPT-2
        for n, p in self.named_parameters():
            if p.dim() == 2:
                small = n.endswith(("proj.weight", "out.weight"))
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.layers) if small else 0.02)
            elif n.endswith("bias"):
                nn.init.zeros_(p)

    def forward(self, idx, targets):
        x = self.wte(idx) + self.wpe(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks:
            x = checkpoint(b, x, use_reentrant=False) if self.ckpt else b(x)
        h, y = self.ln_f(x).flatten(0, 1), targets.flatten()
        if not self.chunks:
            return F.cross_entropy(self.head(h).float(), y)
        # Chunked cross-entropy: each chunk's logits are recomputed in backward, so one chunk is alive at a time
        parts = [checkpoint(chunk_loss, hc, self.head.weight, yc, use_reentrant=False)
                 for hc, yc in zip(h.chunk(self.chunks), y.chunk(self.chunks))]
        return sum(parts) / y.numel()


In [ ]:
import torch
from gpt import GPT, GPTConfig

cap = torch.cuda.get_device_capability()
dtype = torch.bfloat16 if cap >= (8, 0) else torch.float16   # pick by capability, not by is_bf16_supported()
print(torch.cuda.get_device_name(), cap, "| is_bf16_supported():", torch.cuda.is_bf16_supported(), "| using", dtype)

torch.manual_seed(0)
model = GPT(GPTConfig()).cuda()
N = sum(p.numel() for p in model.parameters())
opt = torch.optim.AdamW(model.parameters(), lr=6e-4, fused=True)
scaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)
x = torch.randint(0, 50257, (1, 64), device="cuda")
for _ in range(2):
    with torch.autocast("cuda", dtype=dtype):
        loss = model(x, x)
    scaler.scale(loss).backward()      # no zero_grad: params, grads and Adam state all stay alive
    scaler.step(opt)
    scaler.update()

GB = 1e9
alloc = torch.cuda.memory_allocated()
print(f"parameters {N:,}")
print(f"allocated after a step {alloc / GB:.3f} GB = {alloc / N:.2f} bytes per parameter")
print(f"16 bytes per parameter {16 * N / GB:.3f} GB")
print(f"reserved by the caching allocator {torch.cuda.memory_reserved() / GB:.3f} GB")
del model, opt, scaler, loss, x   # loss.grad_fn still points at every parameter and its grad
torch.cuda.empty_cache()


In [ ]:
# Peak memory of one fp16 autocast + fused AdamW step of GPT-2 124M, from the tensors autograd keeps.
N, L, d, V = 124_439_808, 12, 768, 50257
GB = 1e9

def predict_peak_gb(b, s, ckpt=False, chunks=0):
    tokens = b * s
    layer = 4 * d if ckpt else 38 * d     # bytes per token per block kept for backward
    head = 8 * d if chunks else 6 * d     # final LayerNorm input, plus the hidden state the head reads
    logits = 12 * V / max(chunks, 1)      # fp32 log-probs, their gradient and the logits gradient
    during_loss = 14 * N + tokens * (L * layer + head + logits)   # params 4, Adam 8, fp16 weight copies 2
    end_of_backward = 16 * N + 4 * V * d     # params 4, grads 4, Adam 8, plus the dense embedding gradient
    return max(during_loss, end_of_backward) / GB

for b, s in [(1, 256), (8, 1024), (12, 1024), (16, 1024)]:
    print(f"b={b:>2} s={s:>4}  predicted peak {predict_peak_gb(b, s):5.2f} GB")
print(f"per token: blocks {L * 38 * d:,} B, logits {12 * V:,} B")


In [ ]:
import gc
import torch
from gpt import GPT, GPTConfig

cfg = GPTConfig()

def train_step(model, opt, scaler, b, s, dtype=torch.float16):
    x = torch.randint(0, cfg.vocab, (b, s), device="cuda")
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=dtype, enabled=dtype != torch.float32):
        loss = model(x, x)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    return loss

def measure_peak(b, s, **switches):
    """Peak allocated and reserved GB of the third training step, or None on out-of-memory."""
    model = GPT(cfg, **switches).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=6e-4, fused=True)
    scaler = torch.amp.GradScaler("cuda")
    result = None
    try:
        for i in range(3):
            if i == 2:
                torch.cuda.reset_peak_memory_stats()   # Adam state exists from step 1 on
            train_step(model, opt, scaler, b, s)
        torch.cuda.synchronize()
        result = (torch.cuda.max_memory_allocated() / GB, torch.cuda.max_memory_reserved() / GB)
    except torch.cuda.OutOfMemoryError:
        pass
    del model, opt, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return result

grid = []
for s in (256, 512, 1024):
    for b in (1, 2, 4, 8, 12, 16):
        pred, got = predict_peak_gb(b, s), measure_peak(b, s)
        grid.append((b, s, pred, got))
        if got is None:
            print(f"b={b:>2} s={s:>4}  predicted {pred:5.2f} GB  measured OUT OF MEMORY")
            break
        print(f"b={b:>2} s={s:>4}  predicted {pred:5.2f}  allocated {got[0]:5.2f}  "
              f"reserved {got[1]:5.2f} GB  error {pred / got[0] - 1:+.1%}")


In [ ]:
import time

def tokens_per_sec(b, s, steps=10, dtype=torch.float16, compile=False, **switches):
    model = GPT(cfg, **switches).cuda()
    if compile:
        model = torch.compile(model)
    opt = torch.optim.AdamW(model.parameters(), lr=6e-4, fused=True)
    scaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)
    t = time.perf_counter()
    train_step(model, opt, scaler, b, s, dtype)      # the first step includes any compile time
    torch.cuda.synchronize()
    first = time.perf_counter() - t
    for _ in range(2):
        train_step(model, opt, scaler, b, s, dtype)  # warm-up
    torch.cuda.synchronize()                         # wait for queued GPU work before starting the clock
    t = time.perf_counter()
    for _ in range(steps):
        train_step(model, opt, scaler, b, s, dtype)
    torch.cuda.synchronize()                         # and again before stopping it
    tps = b * s * steps / (time.perf_counter() - t)
    del model, opt, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return tps, first

def mfu(tps, s, peak=65e12):
    flops_per_token = 6 * N + 12 * L * d * s     # 6N for the weights, plus attention scores
    return tps * flops_per_token / peak

tps, first = tokens_per_sec(8, 1024)
print(f"b=8 s=1024  {tps:,.0f} tokens/s  first step {first:.2f} s  MFU {mfu(tps, 1024):.1%} of 65 TFLOPS")

# Timing pieces without synchronize: the clock stops when the CPU has queued the work, not when the GPU has done it
model = GPT(cfg).cuda()
x = torch.randint(0, cfg.vocab, (8, 1024), device="cuda")
for sync in (False, True):
    for _ in range(2):
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(x, x)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(x, x)
        if sync:
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        loss.backward()
        if sync:
            torch.cuda.synchronize()
        t2 = time.perf_counter()
        model.zero_grad(set_to_none=True)
    print(f"synchronize={sync!s:5}  forward {1e3 * (t1 - t0):6.1f} ms  backward {1e3 * (t2 - t1):6.1f} ms")
del model, x, loss
gc.collect()
torch.cuda.empty_cache()


In [ ]:
variants = {
    "baseline": {},
    "checkpointing": {"ckpt": True},
    "chunked CE x8": {"chunks": 8},
    "both": {"ckpt": True, "chunks": 8},
}
cut = {}
for name, sw in variants.items():
    peak = measure_peak(8, 1024, **sw)[0]
    tps, _ = tokens_per_sec(8, 1024, **sw)
    largest = None
    for b in (8, 12, 16, 24, 32, 48, 64, 96, 128):
        if measure_peak(b, 1024, **sw) is None:
            break
        largest = b
    cut[name] = (peak, tps, largest)
    print(f"{name:14} b=8 s=1024  predicted {predict_peak_gb(8, 1024, **sw):5.2f}  measured {peak:5.2f} GB  "
          f"{tps:7,.0f} tokens/s  largest batch at s=1024: {largest}")


In [ ]:
speed = {}
for name, kw in [("fp32", {"dtype": torch.float32}), ("fp16 autocast", {}),
                 ("bf16 autocast", {"dtype": torch.bfloat16}), ("fp16 + torch.compile", {"compile": True})]:
    tps, first = tokens_per_sec(4, 1024, **kw)
    speed[name] = (tps, first)
    print(f"{name:22} {tps:7,.0f} tokens/s  first step {first:6.1f} s")

# fp16 without loss scaling: how many gradient entries in the blocks round to exactly zero?
torch.manual_seed(0)
model = GPT(cfg).cuda()
x = torch.randint(0, cfg.vocab, (8, 1024), device="cuda")
for scale in (1.0, 65536.0):
    model.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16):
        loss = model(x, x)
    (loss * scale).backward()
    g = torch.cat([p.grad.flatten() for p in model.blocks.parameters()])
    print(f"loss scale {scale:>7.0f}  zero gradient entries {(g == 0).float().mean().item():.2%}")
print("smallest fp16 above zero:", torch.finfo(torch.float16).tiny, "normal,", 2.0 ** -24, "subnormal")
del model, x, loss, g
gc.collect()
torch.cuda.empty_cache()


## Part 2. Two GPUs

Each script runs in its own processes under `torchrun`, so the notebook writes it to a file first. Restart the kernel before this part if part 1 ran in the same session, so part 1's tensors do not hold GPU 0's memory.

In [ ]:
%%writefile allreduce.py
# allreduce.py - launch with: torchrun --nproc_per_node=2 allreduce.py
import time
import torch
import torch.distributed as dist

dist.init_process_group("nccl")
rank, n = dist.get_rank(), dist.get_world_size()
torch.cuda.set_device(rank)
for mib in (1, 16, 64, 256, 512):
    x = torch.ones(mib * 2**20 // 4, device="cuda")      # mib MiB of fp32
    for _ in range(3):
        dist.all_reduce(x)                                # warm-up, sets up the NCCL channels
    torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(10):
        dist.all_reduce(x)
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t) / 10
    algbw = x.numel() * 4 / dt / 1e9                      # message bytes per second
    busbw = algbw * 2 * (n - 1) / n                       # bytes each GPU actually sent per second
    if rank == 0:
        print(f"{mib:>4} MiB  {1e3 * dt:7.2f} ms  algbw {algbw:5.2f} GB/s  busbw {busbw:5.2f} GB/s")
dist.destroy_process_group()


In [ ]:
!torchrun --nproc_per_node=2 allreduce.py


In [ ]:
%%writefile ddp.py
# ddp.py - launch with: torchrun --nproc_per_node=2 ddp.py [--sync-every-micro] [--same-seed]
import contextlib, sys, time
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from gpt import GPT, GPTConfig

dist.init_process_group("nccl")
rank, world = dist.get_rank(), dist.get_world_size()
torch.cuda.set_device(rank)
torch.manual_seed(0)                                   # same init everywhere; DDP also broadcasts rank 0's weights
model = DDP(GPT(GPTConfig()).cuda(), device_ids=[rank])
opt = torch.optim.AdamW(model.parameters(), lr=6e-4, fused=True)
scaler = torch.amp.GradScaler("cuda")
data_seed = 1234 if "--same-seed" in sys.argv else 1234 + rank     # each rank must draw different data
gen = torch.Generator(device="cuda").manual_seed(data_seed)
B, S, ACCUM, STEPS = 8, 1024, 4, 10

for step in range(STEPS + 3):
    if step == 3:                                      # 3 warm-up steps, then start the clock
        torch.cuda.synchronize()
        t = time.perf_counter()
    for i in range(ACCUM):
        x = torch.randint(0, 50257, (B, S), device="cuda", generator=gen)
        last = i == ACCUM - 1
        # no_sync skips the gradient all-reduce; only the last micro-batch communicates
        ctx = contextlib.nullcontext() if last or "--sync-every-micro" in sys.argv else model.no_sync()
        with ctx:
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(x, x) / ACCUM
            scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
torch.cuda.synchronize()
tps = STEPS * ACCUM * B * S * world / (time.perf_counter() - t)
losses = [None] * world
dist.all_gather_object(losses, round(loss.item() * ACCUM, 6))
if rank == 0:
    print(f"GPUs {world}  {' '.join(sys.argv[1:]) or 'default'}  {tps:,.0f} tokens/s  last loss per rank {losses}")
dist.destroy_process_group()


In [ ]:
!torchrun --nproc_per_node=1 ddp.py
!torchrun --nproc_per_node=2 ddp.py
!torchrun --nproc_per_node=2 ddp.py --sync-every-micro
!torchrun --nproc_per_node=2 ddp.py --same-seed


In [ ]:
%%writefile fsdp.py
# fsdp.py - launch with: torchrun --nproc_per_node=2 fsdp.py  (or 1, to see one T4 run out of memory)
import time
import torch
import torch.distributed as dist
from torch.distributed.fsdp import fully_shard, MixedPrecisionPolicy
from gpt import GPT, GPTConfig

CFG = GPTConfig(layers=24, heads=16, d=2048)          # about 1.3B parameters
GB = 1e9

def build():
    torch.manual_seed(0)
    model = GPT(CFG, ckpt=True, chunks=8)              # built full size on the CPU
    mp = MixedPrecisionPolicy(param_dtype=torch.float16, reduce_dtype=torch.float32)
    for block in model.blocks:
        fully_shard(block, mp_policy=mp)               # each block is gathered, used and freed as one unit
    fully_shard(model, mp_policy=mp)                   # embeddings, final norm and the tied head
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    return model, opt

def train(model, opt, scaler, steps, gen, B=2, S=1024):
    for _ in range(steps):
        x = torch.randint(0, CFG.vocab, (B, S), device="cuda", generator=gen)
        loss = model(x, x)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
    return loss.item()

if __name__ == "__main__":
    dist.init_process_group("nccl")
    rank, world = dist.get_rank(), dist.get_world_size()
    torch.cuda.set_device(rank)
    model, opt = build()
    N = sum(p.numel() for p in model.parameters())
    scaler = torch.amp.GradScaler("cuda")
    gen = torch.Generator(device="cuda").manual_seed(100 + rank)
    try:
        train(model, opt, scaler, 2, gen)
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        t = time.perf_counter()
        loss = train(model, opt, scaler, 5, gen)
        torch.cuda.synchronize()
        tps = 5 * 2 * 1024 * world / (time.perf_counter() - t)
        print(f"rank {rank} of {world}  N {N:,}  peak {torch.cuda.max_memory_allocated() / GB:.2f} GB  "
              f"16N/{world} = {16 * N / world / GB:.2f} GB  {tps:,.0f} tokens/s  loss {loss:.3f}")
    except torch.cuda.OutOfMemoryError:
        print(f"rank {rank} of {world}  N {N:,}  OUT OF MEMORY at {torch.cuda.memory_allocated() / GB:.2f} GB allocated, "
              f"16N = {16 * N / GB:.2f} GB")
    dist.destroy_process_group()


In [ ]:
!torchrun --nproc_per_node=1 fsdp.py
!torchrun --nproc_per_node=2 fsdp.py


In [ ]:
%%writefile fsdp_ckpt.py
# fsdp_ckpt.py - launch with: torchrun --nproc_per_node=2 fsdp_ckpt.py
import gc, os, time
import torch
import torch.distributed as dist
import torch.distributed.checkpoint as dcp
from torch.distributed.checkpoint.state_dict import get_state_dict, set_state_dict
from fsdp import build, train, CFG
from gpt import GPT

dist.init_process_group("nccl")
rank = dist.get_rank()
torch.cuda.set_device(rank)
PATH = "/tmp/ckpt_fsdp"

model, opt = build()
scaler = torch.amp.GradScaler("cuda")
train(model, opt, scaler, 3, torch.Generator(device="cuda").manual_seed(100 + rank))

# Wrong. Rank 0 alone saves its state_dict. Under FSDP every tensor in it holds only rank 0's shard.
if rank == 0:
    torch.save(model.state_dict(), "/tmp/rank0_only.pt")

# Right. Every rank writes its own shards, model and optimizer, plus the loss scaler.
t = time.perf_counter()
msd, osd = get_state_dict(model, opt)
dcp.save({"model": msd, "optim": osd, "scaler": scaler.state_dict()}, checkpoint_id=PATH)
dist.barrier()
save_s = time.perf_counter() - t
loss_a = train(model, opt, scaler, 2, torch.Generator(device="cuda").manual_seed(7 + rank))
del model, opt, msd, osd
gc.collect()
torch.cuda.empty_cache()

# A fresh model and optimizer load the checkpoint, then take the same 2 steps on the same batches.
model, opt = build()
t = time.perf_counter()
msd, osd = get_state_dict(model, opt)
state = {"model": msd, "optim": osd, "scaler": scaler.state_dict()}
dcp.load(state, checkpoint_id=PATH)
set_state_dict(model, opt, model_state_dict=state["model"], optim_state_dict=state["optim"])
scaler.load_state_dict(state["scaler"])
load_s = time.perf_counter() - t
loss_b = train(model, opt, scaler, 2, torch.Generator(device="cuda").manual_seed(7 + rank))

if rank == 0:
    files = {f: round(os.path.getsize(os.path.join(PATH, f)) / 1e9, 2) for f in sorted(os.listdir(PATH))}
    print("sharded checkpoint files (GB):", files)
    print(f"save {save_s:.1f} s  load {load_s:.1f} s  loss after 2 more steps: continued {loss_a:.6f}  reloaded {loss_b:.6f}")
    sd = torch.load("/tmp/rank0_only.pt", weights_only=False)
    kept = sum(v.to_local().numel() for v in sd.values())
    print(f"rank-0-only file {os.path.getsize('/tmp/rank0_only.pt') / 1e9:.2f} GB, holds {kept:,} values for a model of "
          f"{sum(v.numel() for v in sd.values()):,}")
    try:
        GPT(CFG).load_state_dict({k: v.to_local() for k, v in sd.items()})
    except RuntimeError as e:
        first = [line.strip() for line in str(e).splitlines() if "size mismatch" in line][:1]
        print("loading it into a full model:", first or str(e)[:300])
dist.destroy_process_group()


In [ ]:
!torchrun --nproc_per_node=2 fsdp_ckpt.py
!rm -rf /tmp/ckpt_fsdp /tmp/rank0_only.pt


## Part 3. When training blows up

In [ ]:
!pip install -q datasets tokenizers


In [ ]:
%%writefile prep.py
# prep.py - a 4,096-token byte-level BPE on TinyStories, and the token files sweep.py reads.
import time
import numpy as np
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

ds = load_dataset("roneneldan/TinyStories")
train_text, val_text = ds["train"]["text"][:200_000], ds["validation"]["text"]
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=4096, special_tokens=["<|eot|>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
t = time.time()
tok.train_from_iterator(train_text[:100_000], trainer=trainer)       # train split only
eot = tok.token_to_id("<|eot|>")

def encode(texts):
    ids = []
    for e in tok.encode_batch(texts):
        ids.extend(e.ids)
        ids.append(eot)
    return np.array(ids, dtype=np.uint16)

np.save("tok_train.npy", encode(train_text))
np.save("tok_val.npy", encode(val_text))
print(f"train {np.load('tok_train.npy').size:,} tokens, val {np.load('tok_val.npy').size:,} tokens, {time.time() - t:.0f} s")


In [ ]:
%%writefile tiny.py
# tiny.py - the lltTinyStories GPT (RMSNorm, RoPE, SwiGLU, tied embeddings) with two fixes and two probes.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.w, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)


def rope_tables(T, hd, device):
    inv = 1.0 / (10000 ** (torch.arange(0, hd, 2, device=device).float() / hd))
    f = torch.outer(torch.arange(T, device=device).float(), inv)
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).type_as(x)


class Block(nn.Module):
    def __init__(self, d, h, qk_norm):
        super().__init__()
        self.h = h
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.qkv, self.o = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d, bias=False)
        hid = int(8 * d / 3 / 64 + 0.999) * 64
        self.w1, self.w3, self.w2 = nn.Linear(d, hid, bias=False), nn.Linear(d, hid, bias=False), nn.Linear(hid, d, bias=False)
        self.qn = RMSNorm(d // h) if qk_norm else None      # QK-norm, one RMSNorm over each head's queries
        self.kn = RMSNorm(d // h) if qk_norm else None      # and one over its keys
        self.max_logit = None

    def forward(self, x, cos, sin, probe=False):
        B, T, C = x.shape
        q, k, v = self.qkv(self.n1(x)).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))
        if self.qn is not None:
            q, k = self.qn(q), self.kn(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if probe:   # largest attention logit before softmax, computed in fp32
            s = q.float() @ k.float().transpose(-2, -1) / math.sqrt(q.shape[-1])
            causal = torch.ones(T, T, dtype=torch.bool, device=x.device).triu(1)
            self.max_logit = s.masked_fill(causal, float("-inf")).max().item()
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.o(a.transpose(1, 2).reshape(B, T, C))
        h = self.n2(x)
        return x + self.w2(F.silu(self.w1(h)) * self.w3(h))


class GPT(nn.Module):
    def __init__(self, V=4096, d=320, L=8, H=8, qk_norm=False):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.blocks = nn.ModuleList([Block(d, H, qk_norm) for _ in range(L)])
        self.norm, self.head, self.H = RMSNorm(d), nn.Linear(d, V, bias=False), H
        self.head.weight = self.emb.weight
        for n, p in self.named_parameters():
            if p.dim() == 2:
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * L) if n.endswith(("o.weight", "w2.weight")) else 0.02)

    def forward(self, idx, targets, z_loss=0.0, probe=False):
        cos, sin = rope_tables(idx.shape[1], self.emb.embedding_dim // self.H, idx.device)
        x = self.emb(idx)
        for b in self.blocks:
            x = b(x, cos, sin, probe)
        logits = self.head(self.norm(x)).float()
        loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        if z_loss or probe:
            log_z = torch.logsumexp(logits, dim=-1)          # log of the softmax normalizer, per token
            if z_loss:
                loss = loss + z_loss * log_z.pow(2).mean()  # z-loss pulls log Z toward 0
            if probe:
                self.stats = {"maxAttnLogit": max(b.max_logit for b in self.blocks),
                              "meanLogZ": log_z.mean().item(), "maxAbsLogit": logits.abs().max().item()}
        return loss


In [ ]:
%%writefile sweep.py
# sweep.py - one short TinyStories run per learning rate, with and without the fixes.
# Usage: python sweep.py <size> <variant> <lr> [<lr> ...]   size: 10M or 27M, variant: base, qk or z
import json, math, sys, time
import numpy as np
import torch
from tiny import GPT

SIZES = {"10M": dict(d=320, L=8, H=8), "27M": dict(d=512, L=8, H=8)}
STEPS, WARMUP, B, T = 600, 100, 32, 256
train_ids = np.load("tok_train.npy", mmap_mode="r")
val_ids = np.load("tok_val.npy", mmap_mode="r")

def batch(arr, rng):
    ix = rng.integers(0, arr.size - T - 1, B)
    x = np.stack([arr[i:i + T] for i in ix]).astype(np.int64)
    y = np.stack([arr[i + 1:i + T + 1] for i in ix]).astype(np.int64)
    return torch.from_numpy(x).cuda(), torch.from_numpy(y).cuda()

@torch.no_grad()
def evaluate(model):
    model.eval()
    rng, losses = np.random.default_rng(0), []           # the same 20 validation batches every time
    for i in range(20):
        x, y = batch(val_ids, rng)
        with torch.autocast("cuda", dtype=torch.float16):
            losses.append(model(x, y, probe=i == 0).item())
    model.train()
    return float(np.mean(losses)), model.stats

def run(size, variant, lr, seed=0):
    torch.manual_seed(seed)
    model = GPT(qk_norm=variant == "qk", **SIZES[size]).cuda()
    z = 1e-4 if variant == "z" else 0.0
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda")
    rng, log, skipped = np.random.default_rng(seed + 1), [], 0
    for step in range(STEPS + 1):
        if step % 150 == 0:
            vl, stats = evaluate(model)
            log.append({"step": step, "valLoss": round(vl, 4), **{k: round(v, 3) for k, v in stats.items()}})
            print(size, variant, lr, log[-1], flush=True)
        if step == STEPS:
            break
        frac = (step - WARMUP) / (STEPS - WARMUP)
        lr_now = lr * (step + 1) / WARMUP if step < WARMUP else lr * (0.1 + 0.45 * (1 + math.cos(math.pi * frac)))
        for g in opt.param_groups:
            g["lr"] = lr_now
        x, y = batch(train_ids, rng)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(x, y, z_loss=z)
        scaler.scale(loss).backward()
        scale = scaler.get_scale()
        scaler.step(opt)
        scaler.update()
        skipped += scaler.get_scale() < scale                # the scaler lowers its scale when it skips a step
        opt.zero_grad(set_to_none=True)
    params = sum(p.numel() for p in model.parameters())
    return {"size": size, "params": params, "variant": variant, "lr": lr, "seed": seed, "skippedSteps": int(skipped),
            "final": log[-1], "metrics": log}

if __name__ == "__main__":
    size, variant, lrs = sys.argv[1], sys.argv[2], [float(a) for a in sys.argv[3:]]
    for lr in lrs:
        t = time.time()
        r = run(size, variant, lr)
        r["wallClockSec"] = round(time.time() - t, 1)
        with open(f"p3_{size}_{variant}_{lr:g}.json", "w") as f:
            json.dump(r, f)


In [ ]:
!python prep.py
!CUDA_VISIBLE_DEVICES=0 python sweep.py 10M base 3e-3 1e-2 3e-2 & CUDA_VISIBLE_DEVICES=1 python sweep.py 10M qk 3e-3 1e-2 3e-2 & wait


In [ ]:
import glob, json, math

runs = [json.load(open(f)) for f in glob.glob("p3_*.json")]
table = {}
for r in runs:
    table.setdefault((r["size"], r["variant"]), {})[r["lr"]] = r
for (size, variant), by_lr in sorted(table.items()):
    lrs = sorted(by_lr)
    final = [by_lr[lr]["final"] for lr in lrs]
    start = [by_lr[lr]["metrics"][0]["valLoss"] for lr in lrs]
    # A run that diverged counts as its step-0 loss, as in Wortsman et al. 2023
    clipped = [min(f["valLoss"], s) if math.isfinite(f["valLoss"]) else s for f, s in zip(final, start)]
    best = min(clipped)
    sensitivity = sum(c - best for c in clipped) / len(clipped)
    good = [lr for lr, c in zip(lrs, clipped) if c <= best + 0.1]
    print(f"{size} {variant:4} val loss  " + "  ".join(f"{lr:g}={c:.3f}" for lr, c in zip(lrs, clipped)))
    print(f"         max attention logit  " + "  ".join(f"{f['maxAttnLogit']:.0f}" for f in final))
    print(f"         mean log Z  " + "  ".join(f"{f['meanLogZ']:.1f}" for f in final))
    print(f"         LR sensitivity {sensitivity:.3f}   within 0.1 nats of best from {min(good):g} to {max(good):g}")
